# ADY201m — Cơ sở dữ liệu SQL và truy vấn bằng Python

**Phương án thực hiện:** SQLite lưu bằng tệp và notebook chạy cục bộ, theo phương án thay thế IBM Cloud đã được giảng viên chấp nhận.

Notebook tạo hoặc mở `data/cardio_train.sqlite`, lưu bảng `CARDIO_TRAIN` gồm 70.000 hồ sơ và thực thi 13 SELECT. Q01–Q10 tương ứng 10 bài truy vấn API; Q11–Q13 bổ sung các yêu cầu trên console. Cơ sở dữ liệu tiếp tục tồn tại sau khi đóng kết nối và khi kết thúc phiên notebook.

Các nhóm 10 hồ sơ dùng thứ tự xác định. Tỷ lệ của các nhóm này chỉ mô tả mẫu truy vấn. Nhánh IBM Db2 được giữ để tham khảo hoặc chạy tùy chọn với `RUN_DB2=1`; kết quả đính kèm là kết quả SQLite thực tế.

In [1]:
from pathlib import Path
import os, json, sqlite3, re
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
RAW=ROOT/'data/cardio_train_raw.csv'
if not RAW.exists(): RAW=ROOT/'cardio_train_raw.csv'
df=pd.read_csv(RAW,sep=';')
assert df.shape==(70000,13)
RUN_DB2=os.getenv('RUN_DB2','0')=='1'
db2_conn=None
if RUN_DB2:
    import ibm_db
    keys=['DB2_DATABASE','DB2_HOSTNAME','DB2_PORT','DB2_UID','DB2_PWD']
    missing=[k for k in keys if not os.getenv(k)]
    if missing: raise ValueError('Thiếu biến môi trường: '+', '.join(missing))
    values={k:os.environ[k] for k in keys}
    if any(';' in v for v in values.values()): raise ValueError('Thông tin DSN không được chứa dấu ;')
    dsn=('DATABASE={DB2_DATABASE};HOSTNAME={DB2_HOSTNAME};PORT={DB2_PORT};'
         'PROTOCOL=TCPIP;UID={DB2_UID};PWD={DB2_PWD};SECURITY=SSL;').format(**values)
    if os.getenv('DB2_SSL_CERT'): dsn+='SSLServerCertificate='+os.environ['DB2_SSL_CERT']+';'
    db2_conn=ibm_db.connect(dsn,'','')
    ENGINE='IBM Db2 Cloud — kết nối thật'
    def query(sql):
        statement=ibm_db.exec_immediate(db2_conn,sql)
        try:
            rows=[]
            row=ibm_db.fetch_assoc(statement)
            while row is not False:
                rows.append(row); row=ibm_db.fetch_assoc(statement)
            return pd.DataFrame(rows)
        finally: ibm_db.free_stmt(statement)
else:
    from sqlite_storage import open_cardio_database
    DATABASE_PATH=Path(os.getenv('SQLITE_DB_PATH', str(ROOT/'data/cardio_train.sqlite'))).resolve()
    local, created=open_cardio_database(df, DATABASE_PATH, ROOT/'sql/03_create_table_sqlite.sql')
    ENGINE='SQLite — cơ sở dữ liệu lưu bằng tệp'
    print('Đã tạo và nhập dữ liệu.' if created else 'Đã mở cơ sở dữ liệu có sẵn; dữ liệu khớp CSV gốc.')
    print('Tệp cơ sở dữ liệu:', DATABASE_PATH.relative_to(ROOT) if DATABASE_PATH.is_relative_to(ROOT) else DATABASE_PATH.name)
    def query(sql):
        sql=re.sub(r'FETCH FIRST (\d+) ROWS ONLY',r'LIMIT \1',sql,flags=re.I)
        return pd.read_sql_query(sql,local)
print(ENGINE)

Đã mở cơ sở dữ liệu có sẵn; dữ liệu khớp CSV gốc.
Tệp cơ sở dữ liệu: data\cardio_train.sqlite
SQLite — cơ sở dữ liệu lưu bằng tệp


## Tạo bảng và lưu dữ liệu

`sql/03_create_table_sqlite.sql` định nghĩa bảng SQLite với khóa chính `ID` và ràng buộc cho các mã phân loại. Cell thiết lập nhập CSV gốc khi bảng chưa tồn tại; những lần chạy sau kiểm tra và sử dụng lại dữ liệu, không xóa hay nhân đôi các hồ sơ.

`sql/04_queries_sqlite.sql` chứa 13 truy vấn có thể chạy trực tiếp trong công cụ SQLite. Tuổi được lưu theo ngày và chuyển sang năm trong SELECT.

Hai tệp `sql/01_create_table.sql` và `sql/02_queries.sql` là phiên bản Db2 tham khảo; việc thực thi nhánh Db2 cần dịch vụ và thông tin kết nối riêng.

## Q01 — Tổng số hồ sơ
```sql
SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN;
```

In [2]:
result=query('SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

,TOTAL_ROWS
0,70000


## Q02 — Số người mắc bệnh
```sql
SELECT COUNT(*) AS DISEASE_CASES FROM CARDIO_TRAIN WHERE CARDIO=1;
```

In [3]:
result=query('SELECT COUNT(*) AS DISEASE_CASES FROM CARDIO_TRAIN WHERE CARDIO=1')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

,DISEASE_CASES
0,34979


## Q03 — Số người theo giới tính
```sql
SELECT GENDER, COUNT(*) AS PEOPLE FROM CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER;
```

In [4]:
result=query('SELECT GENDER, COUNT(*) AS PEOPLE FROM CARDIO_TRAIN GROUP BY GENDER ORDER BY GENDER')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

,GENDER,PEOPLE
0,1,45530
1,2,24470


## Q04 — 10 người có AP_HI cao nhất
```sql
SELECT ID, AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC, ID ASC LIMIT 10;
```

In [5]:
result=query('SELECT ID, AP_HI, CARDIO FROM CARDIO_TRAIN ORDER BY AP_HI DESC, ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 8/10; tỷ lệ: 80.0%


,ID,AP_HI,CARDIO
0,58374,16020,1
1,36339,14020,1
2,36414,14020,1
3,66998,14020,1
4,67502,14020,0
5,79116,13010,1
6,79679,13010,0
7,11089,11500,1
8,73356,11020,1
9,99089,2000,1


## Q05 — 10 người có cân nặng trên trung bình
```sql
SELECT ID, WEIGHT, CARDIO FROM CARDIO_TRAIN WHERE WEIGHT>(SELECT AVG(WEIGHT) FROM CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID ASC LIMIT 10;
```

In [6]:
result=query('SELECT ID, WEIGHT, CARDIO FROM CARDIO_TRAIN WHERE WEIGHT>(SELECT AVG(WEIGHT) FROM CARDIO_TRAIN) ORDER BY WEIGHT DESC, ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 8/10; tỷ lệ: 80.0%


,ID,WEIGHT,CARDIO
0,618,200.0,0
1,71945,200.0,1
2,87498,183.0,1
3,52564,181.0,1
4,6716,180.0,1
5,64829,180.0,1
6,86506,180.0,1
7,87612,180.0,1
8,28683,178.0,0
9,39156,178.0,1


## Q06 — 10 người có đường huyết cao
```sql
SELECT ID, GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC>1 ORDER BY ID ASC LIMIT 10;
```

In [7]:
result=query('SELECT ID, GLUC, CARDIO FROM CARDIO_TRAIN WHERE GLUC>1 ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 5/10; tỷ lệ: 50.0%


,ID,GLUC,CARDIO
0,8,2,0
1,12,3,1
2,27,3,0
3,32,2,1
4,40,3,0
5,68,3,1
6,71,2,0
7,92,2,1
8,103,2,0
9,117,2,1


## Q07 — 10 người lớn tuổi nhất
```sql
SELECT ID, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC, ID ASC LIMIT 10;
```

In [8]:
result=query('SELECT ID, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, CARDIO FROM CARDIO_TRAIN ORDER BY AGE DESC, ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 9/10; tỷ lệ: 90.0%


,ID,AGE_YEARS,CARDIO
0,81643,64,1
1,97124,64,1
2,72373,64,1
3,29892,64,1
4,52297,64,1
5,53805,64,1
6,57321,64,0
7,98749,64,1
8,25799,64,1
9,82562,64,1


## Q08 — 10 người có cholesterol mức 3
```sql
SELECT ID, CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL=3 ORDER BY ID ASC LIMIT 10;
```

In [9]:
result=query('SELECT ID, CHOLESTEROL, CARDIO FROM CARDIO_TRAIN WHERE CHOLESTEROL=3 ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 9/10; tỷ lệ: 90.0%


,ID,CHOLESTEROL,CARDIO
0,1,3,1
1,2,3,1
2,9,3,0
3,12,3,1
4,35,3,1
5,42,3,1
6,43,3,1
7,46,3,1
8,59,3,1
9,68,3,1


## Q09 — 10 người hút thuốc
```sql
SELECT ID, SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE=1 ORDER BY ID ASC LIMIT 10;
```

In [10]:
result=query('SELECT ID, SMOKE, CARDIO FROM CARDIO_TRAIN WHERE SMOKE=1 ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 7/10; tỷ lệ: 70.0%


,ID,SMOKE,CARDIO
0,23,1,0
1,29,1,0
2,52,1,1
3,73,1,1
4,74,1,1
5,90,1,1
6,138,1,0
7,140,1,1
8,142,1,1
9,166,1,1


## Q10 — 10 người có vận động
```sql
SELECT ID, ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE=1 ORDER BY ID ASC LIMIT 10;
```

In [11]:
result=query('SELECT ID, ACTIVE, CARDIO FROM CARDIO_TRAIN WHERE ACTIVE=1 ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 3/10; tỷ lệ: 30.0%


,ID,ACTIVE,CARDIO
0,0,1,0
1,1,1,1
2,3,1,1
3,9,1,0
4,12,1,1
5,13,1,0
6,15,1,0
7,16,1,0
8,21,1,0
9,23,1,0


## Q11 — 10 người có AP_LO thấp nhất
```sql
SELECT ID, AP_LO, CARDIO FROM CARDIO_TRAIN ORDER BY AP_LO ASC, ID ASC LIMIT 10;
```

In [12]:
result=query('SELECT ID, AP_LO, CARDIO FROM CARDIO_TRAIN ORDER BY AP_LO ASC, ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 5/10; tỷ lệ: 50.0%


,ID,AP_LO,CARDIO
0,85816,-70,1
1,2845,0,0
2,19258,0,0
3,23512,0,1
4,24837,0,0
5,32749,0,1
6,34120,0,1
7,36325,0,0
8,39577,0,1
9,45400,0,0


## Q12 — 10 người uống rượu
```sql
SELECT ID, ALCO, CARDIO FROM CARDIO_TRAIN WHERE ALCO=1 ORDER BY ID ASC LIMIT 10;
```

In [13]:
result=query('SELECT ID, ALCO, CARDIO FROM CARDIO_TRAIN WHERE ALCO=1 ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 5/10; tỷ lệ: 50.0%


,ID,ALCO,CARDIO
0,23,1,0
1,81,1,0
2,134,1,1
3,136,1,0
4,142,1,1
5,180,1,1
6,199,1,0
7,213,1,1
8,240,1,0
9,318,1,1


## Q13 — Xem dữ liệu và đổi tuổi sang năm
```sql
SELECT ID, AGE AS AGE_DAYS, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, GENDER, CARDIO FROM CARDIO_TRAIN ORDER BY ID ASC LIMIT 10;
```

In [14]:
result=query('SELECT ID, AGE AS AGE_DAYS, CAST(AGE/365 AS INTEGER) AS AGE_YEARS, GENDER, CARDIO FROM CARDIO_TRAIN ORDER BY ID ASC FETCH FIRST 10 ROWS ONLY')
display(result)
if 'CARDIO' in result.columns:
    print(f'Số mắc bệnh: {int(result.CARDIO.sum())}/{len(result)}; tỷ lệ: {result.CARDIO.mean():.1%}')

Số mắc bệnh: 4/10; tỷ lệ: 40.0%


,ID,AGE_DAYS,AGE_YEARS,GENDER,CARDIO
0,0,18393,50,2,0
1,1,20228,55,1,1
2,2,18857,51,1,1
3,3,17623,48,2,1
4,4,17474,47,1,0
5,8,21914,60,1,0
6,9,22113,60,1,0
7,12,22584,61,2,1
8,13,17668,48,1,0
9,14,19834,54,1,0


## Xuất CSV, đóng kết nối và kiểm tra lưu trữ

Dữ liệu xuất giữ tuổi theo ngày để notebook phân tích chỉ chuyển đổi một lần. Sau khi đóng kết nối SQLite, notebook mở lại tệp và kiểm tra số hồ sơ, số người mắc bệnh cùng tính toàn vẹn của cơ sở dữ liệu.

In [15]:
try:
    downloaded=query('SELECT * FROM CARDIO_TRAIN ORDER BY ID')
    downloaded.columns=downloaded.columns.str.lower()
    assert len(downloaded)==70000 and downloaded.id.nunique()==70000
    downloaded.to_csv(ROOT/'data/CARDIO_TRAIN_export.csv',index=False)
    print(f'Đã xuất {len(downloaded):,} dòng. Nguồn: {ENGINE}')
finally:
    if RUN_DB2 and db2_conn is not None:
        import ibm_db
        ibm_db.close(db2_conn)
    elif not RUN_DB2: local.close()
status={'engine':ENGINE,'cloud_executed':RUN_DB2,'rows':len(downloaded),
        'disease_cases':int(downloaded.cardio.sum()),'queries_executed':13,
        'approved_alternative':'SQLite and local notebook execution'}
if not RUN_DB2:
    assert DATABASE_PATH.is_file()
    reopened=sqlite3.connect(DATABASE_PATH)
    try:
        total,cases=reopened.execute('SELECT COUNT(*), SUM(CARDIO) FROM CARDIO_TRAIN').fetchone()
        integrity=reopened.execute('PRAGMA integrity_check').fetchone()[0]
        assert (total,cases,integrity)==(70000,34979,'ok')
        status.update({'persistent':True,
                       'database_file':str(DATABASE_PATH.relative_to(ROOT)) if DATABASE_PATH.is_relative_to(ROOT) else DATABASE_PATH.name,
                       'reopen_verified':True,'integrity_check':integrity})
        print(f'Mở lại tệp: {total:,} hồ sơ, {cases:,} hồ sơ mắc bệnh; integrity_check={integrity}.')
    finally: reopened.close()
(ROOT/'results/db_execution_status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
print('Đã đóng kết nối.')

Đã xuất 70,000 dòng. Nguồn: SQLite — cơ sở dữ liệu lưu bằng tệp
Mở lại tệp: 70,000 hồ sơ, 34,979 hồ sơ mắc bệnh; integrity_check=ok.
Đã đóng kết nối.
